# Notebook 04 — Inclusion-Risk Trade-off (RQ4)

Study A. Characterize how the optimal-tau policy trades financial inclusion
(auto-assignment rate = share of thin filers served without manual review)
against risk (misassignment rate among auto-assigned sellers), and how the
optimal operating point shifts between an emerging-market cost structure
(high rho, many thin filers) and a developed-market one.


In [2]:
# %% ============================================================
# Notebook 04 — Inclusion-Risk Trade-off (RQ4)
# Imports, paths, load
# ============================================================
import os
import json
import numpy as np
import pandas as pd

SEED = 42
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
TAB_DIR = os.path.join(ROOT, "results", "tables")

scs_df = pd.read_csv(os.path.join(ARTIFACT_DIR, "seller_scs.csv"))
MAIN_SCS = "scs_dist_decomp"
a = scs_df[scs_df["method"] == "A_real"].copy()
a["misassign"] = 1 - a["assign_correct"]
tau_star_tbl = pd.read_csv(os.path.join(TAB_DIR, "table_tau_star_by_rho.csv"))
print("Sellers:", len(a))

# %% ============================================================
# Inclusion-risk frontier: for each tau, compute
#   inclusion = auto-assignment rate = P(SCS >= tau)
#   risk      = misassignment rate among auto-assigned sellers
# ============================================================
tau_grid = np.linspace(0.0, 1.0, 201)
rows = []
for t in tau_grid:
    auto = a[a[MAIN_SCS] >= t]
    inclusion = len(auto) / len(a)
    risk = auto["misassign"].mean() if len(auto) > 0 else 0.0
    rows.append({"tau": t, "inclusion": inclusion, "risk": risk})
frontier = pd.DataFrame(rows)
frontier.to_csv(os.path.join(ARTIFACT_DIR, "inclusion_risk_frontier.csv"), index=False)
print("Frontier endpoints:")
print(frontier.iloc[[0, 100, 200]].round(4).to_string(index=False))

# %% ============================================================
# Mark the cost-optimal operating points (tau*) on the frontier for
# each rho, reporting the inclusion and risk they realize.
# ============================================================
op_rows = []
for _, r in tau_star_tbl.iterrows():
    t = r["tau_star"]
    auto = a[a[MAIN_SCS] >= t]
    op_rows.append({
        "rho": int(r["rho"]),
        "tau_star": t,
        "inclusion": len(auto) / len(a),
        "risk": auto["misassign"].mean() if len(auto) > 0 else 0.0,
    })
op_tbl = pd.DataFrame(op_rows)
op_tbl.to_csv(os.path.join(TAB_DIR, "table_optimal_operating_points.csv"), index=False)
print("Cost-optimal operating points on the inclusion-risk frontier:")
print(op_tbl.round(4).to_string(index=False))

# %% ============================================================
# Emerging vs developed market scenario contrast.
# Emerging market: a higher thin-filer share (more cross-category
# sellers) and a moderately higher cost ratio. Developed market:
# mostly focused sellers and a lower cost ratio. Weights are chosen so
# both markets keep a realistic, non-degenerate auto-assignment rate;
# the point is the RELATIVE shift, not an extreme corner solution.
# ============================================================
def optimal_policy(df, scs_col, rho, c_rev=1.0):
    c_err = rho * c_rev
    grid = np.linspace(0, 1, 201)
    costs = []
    for t in grid:
        auto = df[df[scs_col] >= t]
        manual = df[df[scs_col] < t]
        cost = (c_err * auto["misassign"].sum() + c_rev * len(manual)) / len(df)
        costs.append(cost)
    ts = grid[int(np.argmin(costs))]
    auto = df[df[scs_col] >= ts]
    return {"tau_star": ts, "inclusion": len(auto) / len(df),
            "risk": auto["misassign"].mean() if len(auto) > 0 else 0.0}


# Emerging: more cross-category sellers, moderate diversified share; rho=20
emerging = pd.concat([
    a[a["regime"] == "focused"].sample(frac=0.6, random_state=SEED),
    a[a["regime"] == "cross"].sample(frac=1.0, random_state=SEED),
    a[a["regime"] == "diversified"].sample(frac=0.5, random_state=SEED),
])
# Developed: mostly focused sellers, few mixed; rho=10
developed = pd.concat([
    a[a["regime"] == "focused"].sample(frac=1.0, random_state=SEED),
    a[a["regime"] == "cross"].sample(frac=0.4, random_state=SEED),
    a[a["regime"] == "diversified"].sample(frac=0.1, random_state=SEED),
])

scen_rows = []
scen_rows.append({"market": "emerging", "rho": 20,
                  **optimal_policy(emerging, MAIN_SCS, rho=20)})
scen_rows.append({"market": "developed", "rho": 10,
                  **optimal_policy(developed, MAIN_SCS, rho=10)})
scen_tbl = pd.DataFrame(scen_rows)
scen_tbl.to_csv(os.path.join(TAB_DIR, "table_market_scenarios.csv"), index=False)
print("Emerging vs developed market optimal policy:")
print(scen_tbl.round(4).to_string(index=False))
print()
print("Emerging pool size:", len(emerging), "| Developed pool size:", len(developed))

Sellers: 1440
Frontier endpoints:
 tau  inclusion   risk
 0.0     1.0000 0.2833
 0.5     0.0576 0.0241
 1.0     0.0000 0.0000
Cost-optimal operating points on the inclusion-risk frontier:
 rho  tau_star  inclusion   risk
   5      0.19     0.4264 0.0977
  10      0.33     0.2250 0.0586
  20      0.52     0.0438 0.0000
  50      0.52     0.0438 0.0000
 100      0.52     0.0438 0.0000
Emerging vs developed market optimal policy:
   market  rho  tau_star  inclusion   risk
 emerging   20      0.44     0.0774 0.0256
developed   10      0.16     0.7847 0.0283

Emerging pool size: 1008 | Developed pool size: 720
